# Distillation: DeepSeek-V4-Pro (teacher) -> Qwen2.5-0.5B-Instruct (student), GSM8K
**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

Steps: install pinned libs -> write scripts + data -> measure the student BEFORE (1319 test questions) -> train LoRA on the teacher's filtered solutions -> measure AFTER -> download `results.zip`.
Nothing here calls the teacher: its answers are already inside the data zip.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')

In [ ]:
# Colab preinstalls torchao 0.10, which peft 0.21 rejects (needs >=0.16); we don't use it
!pip uninstall -y -q torchao
!pip install -q transformers==5.21.0 peft==0.21.2 accelerate==1.15.0 matplotlib
!mkdir -p data out

In [ ]:
%%writefile common.py
"""Shared by select_questions.py, teacher.py, filter_data.py, generate.py and score.py:
prompt layout, answer parsing, arithmetic check of a written solution."""
import re

INSTRUCTION = (
    "Solve the math problem step by step. Write the reasoning as short plain-text steps "
    "(no markdown, no LaTeX). The last line must be exactly: Answer: <number>"
)

NUM = r"-?\d[\d,]*(?:\.\d+)?"


def build_prompt(question):
    """The same user message for the teacher, the base student and the tuned student."""
    return f"{question.strip()}\n\n{INSTRUCTION}"


def to_float(s):
    try:
        return float(s.replace(",", "").replace("$", "").rstrip("."))
    except ValueError:
        return None


def gold_answer(answer_field):
    """GSM8K reference: the number after '####'."""
    return to_float(answer_field.split("####")[-1].strip())


def same_number(a, b):
    return a is not None and b is not None and abs(a - b) < 1e-6


STRICT = re.compile(rf"Answer: ({NUM})")


def parse_answer(text):
    """Return (number or None, strict_format_ok).

    strict: the last non-empty line is exactly 'Answer: <number>' and that is the only 'Answer:' in the text.
    Lenient fallback (used for accuracy, so a base model is not punished for style alone):
    'Answer: $18', \\boxed{18}, '#### 18', 'the answer is 18', else the last number in the text.
    """
    t = text.strip()
    lines = [l for l in t.splitlines() if l.strip()]
    if lines:
        m = STRICT.fullmatch(lines[-1].strip())
        if m and t.count("Answer:") == 1:
            return to_float(m.group(1)), True
    for pat in (rf"Answer:\s*\**\s*\$?\s*({NUM})", r"\\boxed\{\s*\\?\$?\s*(" + NUM + r")", rf"####\s*\$?\s*({NUM})",
                rf"(?i:answer is)\s*:?\s*\**\$?\s*({NUM})"):
        found = re.findall(pat, t)
        if found:
            return to_float(found[-1]), False
    found = re.findall(NUM, t)
    return (to_float(found[-1]), False) if found else (None, False)


EQ = re.compile(
    rf"((?:\$?\d[\d,]*(?:\.\d+)?[ \t]*[-+−×*/÷x][ \t]*)+\$?\d[\d,]*(?:\.\d+)?)[ \t]*=[ \t]*\$?({NUM})"
    r"(?![\d%A-Za-z]|[.,]\d|[ \t]*[-+−×*/÷=^(%]|[ \t]*x[ \t]*\d|[ \t]+and[ \t]+\d)")


def bad_equations(text):
    """Equations 'a op b [op c] = r' written in the text whose result is arithmetically wrong.

    Only complete, plain-number equations are judged: skipped when the left side continues an expression
    (preceded by an operator, bracket, letter or digit), the result continues (followed by an operator,
    '=', '%'), integer division with a stated remainder, or the sign of the first number was lost. Rounding is tolerated (rel 1% / abs 0.011).
    Returns a list of the offending strings.
    """
    bad = []
    for m in EQ.finditer(text):
        pre = text[:m.start()]
        if re.search(r"[-−+*/×÷^(]\s*$", pre) or re.search(r"[\w).]$", pre) or re.search(
                r"(\d|\b(times|of|by|plus|minus|and))[ \t]+$", pre):
            continue  # continues a longer expression, a mixed fraction ('16 1/2') or prose ('18 times 2/3')
        left = m.group(1).replace("−", "-").replace("×", "*").replace("÷", "/").replace("$", "").replace(",", "")
        left = re.sub(r"(?<=\d)\s*x\s*(?=\d)", "*", left)
        try:
            val = eval(left, {"__builtins__": {}}, {})  # only digits and + - * / survive the regex
        except Exception:
            continue
        rhs = to_float(m.group(2))
        if rhs is not None and "remainder" in text and rhs == int(val):
            continue  # integer division with a remainder
        if rhs is not None and abs(val - rhs) > max(0.011, 0.01 * abs(val)):
            bad.append(m.group(0).strip())
    return bad


def solution_steps(answer_field):
    """Number of reasoning lines in the GSM8K reference solution (difficulty proxy)."""
    return len([l for l in answer_field.split("####")[0].splitlines() if l.strip()])


In [ ]:
%%writefile train_lora.py
#!/usr/bin/env python3
"""Distillation (SFT on teacher answers): LoRA fine-tune of a small chat model on GSM8K solutions written by DeepSeek.

Data: data/train.jsonl + data/valid.jsonl (built by filter_data.py; each row has `messages` = user, assistant).
Only assistant tokens contribute to the loss; prompts are masked with -100.
Writes into --out: adapter/, loss_log.json, run_info.json.
"""
import argparse
import json
import platform
import time
from pathlib import Path

import torch
import transformers
import peft
from peft import LoraConfig, get_peft_model
from transformers import (AutoModelForCausalLM, AutoTokenizer, Trainer,
                          TrainingArguments)


def encode(tok, messages, max_len):
    """Chat-template one (user -> assistant) pair; return ids + labels or None if too long."""
    prompt = tok.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    reply = messages[-1]["content"].strip() + tok.eos_token
    reply_ids = tok(reply, add_special_tokens=False)["input_ids"]
    ids = prompt_ids + reply_ids
    if len(ids) > max_len:
        return None
    return {"input_ids": ids, "labels": [-100] * len(prompt_ids) + reply_ids}


def build_dataset(tok, train_path, valid_path, max_len):
    out, skipped = {}, 0
    for name, path in (("train", train_path), ("valid", valid_path)):
        rows = [json.loads(line)["messages"] for line in open(path, encoding="utf-8")]
        enc = [encode(tok, m, max_len) for m in rows]
        skipped += sum(e is None for e in enc)
        out[name] = [e for e in enc if e is not None]
    return out["train"], out["valid"], skipped


class Collator:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, batch):
        width = max(len(b["input_ids"]) for b in batch)
        ids, labels, mask = [], [], []
        for b in batch:
            pad = width - len(b["input_ids"])
            ids.append(b["input_ids"] + [self.pad_id] * pad)
            labels.append(b["labels"] + [-100] * pad)
            mask.append([1] * len(b["input_ids"]) + [0] * pad)
        return {"input_ids": torch.tensor(ids), "labels": torch.tensor(labels),
                "attention_mask": torch.tensor(mask)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="Qwen/Qwen2.5-0.5B-Instruct")
    ap.add_argument("--out", default="out")
    ap.add_argument("--train", default="data/train.jsonl")
    ap.add_argument("--valid", default="data/valid.jsonl")
    ap.add_argument("--max-len", type=int, default=640)
    ap.add_argument("--epochs", type=float, default=2)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--accum", type=int, default=2)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--rank", type=int, default=64)
    ap.add_argument("--alpha", type=int, default=128)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    cuda = torch.cuda.is_available()
    # T4 (sm_75) has no native bf16 -> fp16 mixed precision; Ampere+ -> bf16.
    use_bf16 = cuda and torch.cuda.get_device_capability()[0] >= 8
    use_fp16 = cuda and not use_bf16
    dtype = torch.bfloat16 if use_bf16 else torch.float16 if use_fp16 else torch.float32
    transformers.set_seed(args.seed)

    tok = AutoTokenizer.from_pretrained(args.model)
    train, val, skipped = build_dataset(tok, args.train, args.valid, args.max_len)
    n_tokens = sum(len(e["input_ids"]) for e in train)
    n_loss_tokens = sum(sum(l != -100 for l in e["labels"]) for e in train)
    print(f"train={len(train)} val={len(val)} skipped_too_long={skipped} "
          f"tokens={n_tokens} loss_tokens={n_loss_tokens}")

    model = AutoModelForCausalLM.from_pretrained(args.model, dtype=dtype)
    model.config.use_cache = False
    model.enable_input_require_grads()  # needed for gradient checkpointing with frozen embeddings
    model = get_peft_model(model, LoraConfig(
        r=args.rank, lora_alpha=args.alpha, lora_dropout=0.05, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    for p in model.parameters():  # fp16 grads cannot be unscaled -> keep trainable LoRA weights in fp32
        if p.requires_grad:
            p.data = p.data.float()
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {n_trainable:,} / {n_total:,} ({100 * n_trainable / n_total:.2f}%)")

    steps_per_epoch = max(1, len(train) // (args.batch * args.accum))
    total_steps = int(steps_per_epoch * args.epochs)
    train_args = TrainingArguments(
        output_dir=str(out / "ckpt"), per_device_train_batch_size=args.batch,
        per_device_eval_batch_size=args.batch, gradient_accumulation_steps=args.accum,
        num_train_epochs=args.epochs, learning_rate=args.lr, lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.05 * total_steps)), weight_decay=0.0,
        logging_steps=5, eval_strategy="steps", eval_steps=max(1, total_steps // 4),
        save_strategy="no", report_to="none", seed=args.seed,
        fp16=use_fp16, bf16=use_bf16, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False, dataloader_num_workers=2 if cuda else 0,
        use_cpu=not cuda)
    trainer = Trainer(model=model, args=train_args, train_dataset=train, eval_dataset=val,
                      data_collator=Collator(tok.pad_token_id))

    base_eval = trainer.evaluate()  # step 0: val loss of the untouched base model (LoRA B=0)
    print("base val loss:", base_eval["eval_loss"])
    if cuda:
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    result = trainer.train()
    train_seconds = time.time() - t0
    final_eval = trainer.evaluate()

    model.save_pretrained(out / "adapter")
    adapter_mb = sum(f.stat().st_size for f in (out / "adapter").glob("*")) / 1e6
    log = [{"step": 0, "eval_loss": base_eval["eval_loss"]}] + trainer.state.log_history
    (out / "loss_log.json").write_text(json.dumps(log, indent=1))
    info = {
        "base_model": args.model, "dataset": "data/train.jsonl (GSM8K questions, solutions by the teacher)", "method": "LoRA (fp16/bf16 base, no quantization)",
        "gpu": torch.cuda.get_device_name(0) if cuda else "cpu",
        "gpu_mem_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if cuda else None,
        "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if cuda else None,
        "precision": str(dtype), "python": platform.python_version(), "torch": torch.__version__,
        "transformers": transformers.__version__, "peft": peft.__version__,
        "n_train": len(train), "n_val": len(val), "skipped_too_long": skipped, "max_len": args.max_len,
        "train_tokens_per_epoch": n_tokens, "loss_tokens_per_epoch": n_loss_tokens,
        "epochs": args.epochs, "per_device_batch": args.batch, "grad_accum": args.accum,
        "effective_batch": args.batch * args.accum, "optimizer_steps": result.global_step,
        "lr": args.lr, "lr_scheduler": "cosine", "warmup_steps": train_args.warmup_steps,
        "lora_r": args.rank, "lora_alpha": args.alpha, "lora_dropout": 0.05,
        "lora_targets": "q,k,v,o,gate,up,down", "trainable_params": n_trainable,
        "total_params": n_total, "seed": args.seed,
        "train_seconds": round(train_seconds, 1), "train_minutes": round(train_seconds / 60, 1),
        "final_train_loss_avg": result.training_loss,
        "val_loss_before": base_eval["eval_loss"], "val_loss_after": final_eval["eval_loss"],
        "adapter_size_mb": round(adapter_mb, 1),
    }
    (out / "run_info.json").write_text(json.dumps(info, indent=1))
    print(json.dumps(info, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile generate.py
#!/usr/bin/env python3
"""Run data/test.jsonl (GSM8K test) through the student before and after distillation (greedy decoding).

Configs: base (untouched student), tuned (student + LoRA adapter, merged into the weights = what you would deploy).
For each config, written to <out>/outputs/<config>.json:
  items   response, new_tokens, truncated, mean_logprob / min_logprob of the generated tokens
  meta    batched run: wall seconds, req/s, generated tokens/s, peak GPU memory
          latency bench: batch=1 on the first --bench-n questions (seconds per request, tokens/s)
Prompt = common.build_prompt(question), identical for teacher, base and tuned.
"""
import argparse
import json
import statistics
import time
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from common import build_prompt


def encode(tok, questions, device):
    texts = [tok.apply_chat_template([{"role": "user", "content": build_prompt(q)}], tokenize=False,
                                     add_generation_prompt=True) for q in questions]
    return tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(device)


@torch.no_grad()
def generate(model, tok, questions, max_new_tokens):
    enc = encode(tok, questions, model.device)
    gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                         top_k=None, repetition_penalty=1.0, pad_token_id=tok.pad_token_id)
    out = []
    for row in gen[:, enc["input_ids"].shape[1]:]:
        ids = [t for t in row.tolist() if t != tok.pad_token_id]
        out.append(ids)
    return out


@torch.no_grad()
def logprobs(model, tok, questions, gen_ids, batch=4):
    """Teacher-forced pass over prompt+reply: per-token log-probability of the generated tokens."""
    res = []
    for i in range(0, len(questions), batch):
        qs, gs = questions[i:i + batch], gen_ids[i:i + batch]
        prompts = [tok(tok.apply_chat_template([{"role": "user", "content": build_prompt(q)}], tokenize=False,
                                               add_generation_prompt=True), add_special_tokens=False)["input_ids"]
                   for q in qs]
        width = max(len(p) + len(g) for p, g in zip(prompts, gs))
        ids = torch.full((len(qs), width), tok.pad_token_id)
        mask = torch.zeros((len(qs), width), dtype=torch.long)
        for k, (p, g) in enumerate(zip(prompts, gs)):
            ids[k, :len(p) + len(g)] = torch.tensor(p + g)
            mask[k, :len(p) + len(g)] = 1
        logits = model(input_ids=ids.to(model.device), attention_mask=mask.to(model.device)).logits
        for k, (p, g) in enumerate(zip(prompts, gs)):
            if not g:
                res.append((0.0, 0.0))
                continue
            lp = torch.log_softmax(logits[k, len(p) - 1:len(p) + len(g) - 1].float(), dim=-1)
            tok_lp = lp.gather(1, torch.tensor(g, device=lp.device)[:, None])[:, 0]
            res.append((tok_lp.mean().item(), tok_lp.min().item()))
        del logits
    return res


def run_config(name, model, tok, rows, args, out_dir):
    cuda = torch.cuda.is_available()
    questions = [r["question"] for r in rows]
    order = sorted(range(len(rows)), key=lambda i: len(rows[i]["question"]))  # similar lengths -> less padding
    gen_ids = [None] * len(rows)
    if cuda:
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    t0 = time.time()
    for s in range(0, len(order), args.batch):
        idx = order[s:s + args.batch]
        for i, g in zip(idx, generate(model, tok, [questions[i] for i in idx], args.max_new_tokens)):
            gen_ids[i] = g
    if cuda:
        torch.cuda.synchronize()
    wall = time.time() - t0
    peak = round(torch.cuda.max_memory_allocated() / 1e9, 2) if cuda else None
    n_tok = sum(len(g) for g in gen_ids)
    print(f"{name}: {len(rows)} questions, {wall:.0f}s, {n_tok / wall:.0f} tok/s, peak {peak} GB", flush=True)

    # latency of a single request (batch = 1), the way an interactive service would see it
    lat = []
    for q in questions[:args.bench_n]:
        if cuda:
            torch.cuda.synchronize()
        t = time.time()
        g = generate(model, tok, [q], args.max_new_tokens)[0]
        if cuda:
            torch.cuda.synchronize()
        lat.append((time.time() - t, len(g)))

    lps = logprobs(model, tok, questions, gen_ids)
    items = []
    for r, g, (mean_lp, min_lp) in zip(rows, gen_ids, lps):
        items.append({"id": r["id"], "response": tok.decode(g, skip_special_tokens=True).strip(),
                      "new_tokens": len(g), "truncated": len(g) >= args.max_new_tokens,
                      "mean_logprob": round(mean_lp, 4), "min_logprob": round(min_lp, 3)})
    meta = {"config": name, "n": len(rows), "batch": args.batch, "max_new_tokens": args.max_new_tokens,
            "gpu": torch.cuda.get_device_name(0) if cuda else "cpu", "peak_gpu_mem_gb": peak,
            "batched_wall_s": round(wall, 1), "batched_req_per_s": round(len(rows) / wall, 3),
            "batched_gen_tok_per_s": round(n_tok / wall, 1), "total_gen_tokens": n_tok,
            "bench_n": len(lat), "latency_s_median": round(statistics.median(l for l, _ in lat), 2),
            "latency_tok_per_s": round(sum(n for _, n in lat) / sum(l for l, _ in lat), 1),
            "dtype": str(model.dtype)}
    (out_dir / f"{name}.json").write_text(json.dumps({"meta": meta, "items": items}, indent=1, ensure_ascii=False))
    print(json.dumps(meta), flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="Qwen/Qwen2.5-0.5B-Instruct")
    ap.add_argument("--adapter", default="out/adapter")
    ap.add_argument("--eval", default="data/test.jsonl")
    ap.add_argument("--out", default="out")
    ap.add_argument("--max-new-tokens", type=int, default=512)
    ap.add_argument("--batch", type=int, default=48)
    ap.add_argument("--bench-n", type=int, default=20)
    ap.add_argument("--limit", type=int, default=0, help="only first N questions (smoke test)")
    ap.add_argument("--only", default="", help="comma list of configs to run (base,tuned)")
    args = ap.parse_args()

    rows = [json.loads(l) for l in open(args.eval, encoding="utf-8")]
    if args.limit:
        rows = rows[:args.limit]
    cuda = torch.cuda.is_available()
    tok = AutoTokenizer.from_pretrained(args.model, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(args.model, dtype=torch.float16 if cuda else torch.float32)
    if cuda:
        model.to("cuda")
    model.eval()
    out_dir = Path(args.out) / "outputs"
    out_dir.mkdir(parents=True, exist_ok=True)
    only = set(filter(None, args.only.split(",")))
    if not only or "base" in only:
        run_config("base", model, tok, rows, args, out_dir)
    if not only or "tuned" in only:
        model = PeftModel.from_pretrained(model, args.adapter).merge_and_unload()
        model.eval()
        run_config("tuned", model, tok, rows, args, out_dir)


if __name__ == "__main__":
    main()


In [ ]:
import base64, io, zipfile
BLOB = '...'
zipfile.ZipFile(io.BytesIO(base64.b64decode(BLOB))).extractall('.')
!wc -l data/*.jsonl

## 1. Student BEFORE distillation (greedy, 1319 questions, + batch=1 latency bench)

In [ ]:
import os, json
!python generate.py --only base --out out
assert os.path.exists('out/outputs/base.json'), 'Generation failed - see the traceback above'

## 2. Train LoRA on the teacher's solutions (`eval_loss` is on data/valid.jsonl)

In [ ]:
!python train_lora.py --out out
assert os.path.exists('out/run_info.json'), 'Training failed - see the traceback above'

In [ ]:
import matplotlib.pyplot as plt
log = json.load(open('out/loss_log.json'))
tr = [(e['step'], e['loss']) for e in log if 'loss' in e]
ev = [(e['step'], e['eval_loss']) for e in log if 'eval_loss' in e]
plt.plot(*zip(*tr), label='train'); plt.plot(*zip(*ev), 'o-', label='valid')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=.3); plt.show()
print(open('out/run_info.json').read())

## 3. Student AFTER distillation (adapter merged into the weights)

In [ ]:
!python generate.py --only tuned --adapter out/adapter --out out
assert os.path.exists('out/outputs/tuned.json'), 'Generation failed - see the traceback above'

## Quick look + download

In [ ]:
from common import parse_answer
gold = {r['id']: r['gold'] for r in map(json.loads, open('data/test.jsonl'))}
for n in ['base', 'tuned']:
    items = json.load(open(f'out/outputs/{n}.json'))['items']
    acc = sum(parse_answer(o['response'])[0] == gold[o['id']] for o in items) / len(items)
    print(n, f'accuracy {acc:.1%}', json.load(open(f'out/outputs/{n}.json'))['meta'])
print(items[0]['response'])

In [ ]:
!cd out && zip -qr ../results.zip adapter outputs loss_log.json run_info.json
from google.colab import files
files.download('results.zip')